In [ ]:
# Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')
FOLDERNAME = 'cs231n/project/'
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

In [ ]:
# Check GPU availability

gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

# Check TensorFlow and PyTorch versions
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))
print("TensorFlow version: ", tf.__version__)

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Clone and install dependencies
%cd /content
!git clone --recursive https://github.com/camenduru/gaussian-splatting
!pip install -q plyfile

%cd /content/gaussian-splatting
!pip install -q /content/gaussian-splatting/submodules/diff-gaussian-rasterization
!pip install -q /content/gaussian-splatting/submodules/simple-knn
!sudo apt-get update
!sudo apt-get install -y colmap
!sudo apt-get install xvfb

In [ ]:
import os
import subprocess
import shutil

# Define paths
project_path = '/content/drive/My Drive/cs231n/project/'
frames_path = project_path + 'framesv4/'
output_base_path = project_path + 'output/'

# Prepare Data for Scene 0
scene_num = '0'
scene_folder = f'scene{scene_num}'
source_path = os.path.join(frames_path, scene_folder)
output_path = os.path.join(output_base_path, scene_folder)

# Ensure the paths are correctly set
print(f"Source path: {source_path}")
print(f"Output path: {output_path}")

# Verify contents of the input path
input_path = source_path
if os.path.exists(input_path) and len(os.listdir(input_path)) > 0:
    print(f"Input directory exists and contains images: {os.listdir(input_path)[:20]}")
else:
    raise Exception(f"Input directory {input_path} does not exist or contains no images.")

In [47]:
# Ensure all necessary directories exist
def ensure_directory_exists(path):
    if not os.path.exists(path):
        os.makedirs(path)

# Define paths
database_path = os.path.join(source_path, 'distorted/database.db')
sparse_path = os.path.join(source_path, 'distorted/sparse')
undistorted_path = os.path.join(source_path, 'undistorted')
undistorted_images_path = os.path.join(undistorted_path)

# Ensure directories exist
ensure_directory_exists(os.path.dirname(database_path))
ensure_directory_exists(sparse_path)
ensure_directory_exists(undistorted_path)

In [ ]:
def run_command(command):
    process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    while True:
        output = process.stdout.readline()
        if output == b'' and process.poll() is not None:
            break
        if output:
            print(output.strip().decode())
    rc = process.poll()
    return rc

In [ ]:
# Feature extraction
feature_extractor_cmd = f"colmap feature_extractor --database_path '{database_path}' --image_path '{input_path}' --ImageReader.single_camera 1"
print(f"Running feature extraction command: {feature_extractor_cmd}")
run_command(feature_extractor_cmd)

In [ ]:
# Feature matching
feature_matching_cmd = f"colmap exhaustive_matcher --database_path '{database_path}' --SiftMatching.use_gpu 1"
print(f"Running feature matching command: {feature_matching_cmd}")
run_command(feature_matching_cmd)

In [ ]:
# Bundle adjustment
mapper_cmd = f"colmap mapper --database_path '{database_path}' --image_path '{input_path}' --output_path '{sparse_path}'"
print(f"Running mapper command: {mapper_cmd}")
run_command(mapper_cmd)

In [ ]:
!{colmap_exe_path}/colmap model_analyzer --path {workspace_path}/distorted/sparse/0

In [ ]:
# Run the dense reconstruction commands
colmap_exe_path = '/usr/bin'
workspace_path = source_path
os.makedirs(workspace_path, exist_ok=True)

def run_colmap_command(command):
    full_command = f"{colmap_exe_path}/{command}"
    process = subprocess.Popen(full_command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    stdout, stderr = process.communicate()
    return stdout.decode(), stderr.decode(), process.returncode

run_colmap_command(f"colmap patch_match_stereo --workspace_path {workspace_path} --workspace_format COLMAP --PatchMatchStereo.max_image_size 2000 --PatchMatchStereo.geom_consistency true")
run_colmap_command(f"colmap stereo_fusion --workspace_path {workspace_path} --workspace_format COLMAP --input_type geometric --output_path {workspace_path}/fused.ply")
run_colmap_command(f"colmap poisson_mesher --input_path {workspace_path}/fused.ply --output_path {workspace_path}/meshed-poisson.ply")
run_colmap_command(f"colmap delaunay_mesher --input_path {workspace_path} --input_type dense --output_path {os.path.join(workspace_path, 'meshed-delaunay.ply')}")

# Verify if .ply files are created
ply_files = [f for f in os.listdir(workspace_path) if f.endswith('.ply')]
if ply_files:
    print("Found .ply files:")
    for file in ply_files:
        print(file)
else:
    print("No .ply files found in the specified directory.")

In [ ]:
# Copy undistorted images and sparse files
if os.path.exists(undistorted_images_path):
    shutil.rmtree(undistorted_images_path)
shutil.copytree(input_path, undistorted_images_path)

if os.path.exists(os.path.join(undistorted_path,'sparse')):
    shutil.rmtree(os.path.join(undistorted_path,'sparse'))
shutil.copytree(sparse_path, os.path.join(undistorted_path,'sparse'))

In [ ]:
# Zip the undistorted directory
zip_filename = os.path.join(output_base_path, f'scene{scene_num}.zip')
print(f"Zipping directory {undistorted_path} to {zip_filename}")
shutil.make_archive(zip_filename.replace('.zip',''), 'zip', undistorted_path)

# Unzip for rendering
unzip_dir = os.path.join(output_base_path, f'scene{scene_num}', 'images')
shutil.unpack_archive(zip_filename, unzip_dir, 'zip')
images_dir = os.path.join(unzip_dir, 'images')

In [ ]:
# Define the output model directory
output_model_dir = os.path.join(output_base_path, f'scene{scene_num}', 'model')

# Ensure the output model directory exists
os.makedirs(output_model_dir, exist_ok=True)

# Run the training script
train_script_path = '/content/gaussian-splatting/train.py'
train_cmd = f"python {train_script_path} -s '{unzip_dir}' -o '{output_model_dir}'"
print(f"Running training command: {train_cmd}")
run_command(train_cmd)

# Run the rendering script
render_script_path = '/content/gaussian-splatting/render.py'
render_cmd = f"python {render_script_path} --model_path '{output_model_dir}' --iteration 30000"
print(f"Running render command: {render_cmd}")
run_command(render_cmd)

In [ ]:
# Define the output video path
output_video_path = os.path.join(output_base_path, f'scene{scene_num}_video.mp4')

# Combine rendered images into a video
ffmpeg_cmd = f"ffmpeg -y -framerate 3 -i '{images_dir}/%05d.png' -vf 'pad=ceil(iw/2)*2:ceil(ih/2)*2' '{output_video_path}'"
print(f"Running ffmpeg command: {ffmpeg_cmd}")
run_command(ffmpeg_cmd)

In [ ]:
!pip install pyngrok

# Start ngrok tunnel and HTTP server
from pyngrok import ngrok

ngrok.set_auth_token("2gWHu0eX53vyboV6uhJrDSIAGWR_6dcTE3DgBLkm3ZLz73svy")

# Start an HTTP tunnel on the default port 7860
public_url = ngrok.connect(7860, "http")
print(f"Ngrok tunnel available at: {public_url}")

# Change directory to the root of your project
%cd /content/gaussian-splatting

# Start the HTTP server on port 7860
!python -m http.server 7860